# Breast cancer ann

Original experiment from [Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow](https://github.com/Muhammad-Huzifa/Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow/blob/dfa17b2b298631b10e5904b2f1c39ad64d424bc1/Pytorch/ANN_Breast_Cancer_DataSet_Using%20pytorch.ipynb). Run from a fresh kernel. See [dataset and environment instructions](../../../docs/DATASETS.md).

Framework training, downloaded datasets, and custom image inputs require the dependencies and resources described in the collection README. This migration has checked syntax and paths; it has not repeated every training experiment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements/base.txt").is_file())
TABULAR = ROOT / "data/tabular"
IMAGES = ROOT / "data/images"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

In [ ]:
df.drop(columns=['Unnamed: 32','id'],inplace=True)

In [ ]:
df.head()

In [ ]:
X_train , X_test ,y_train , y_test = train_test_split(df.iloc[:,1:] , df.iloc[:,0] , random_state=42 , test_size=0.2)
X_train.shape , X_test.shape , y_train.shape , y_test.shape

**Scaling**

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

**Label Encoding**

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
y_train, y_test

**Convert Numpy to Pytorch Tensor**

In [ ]:
# Convert Numpy to Pytorch Tensor
X_train_tensor = torch.from_numpy(X_train).type(torch.float64)
X_test_tensor = torch.from_numpy(X_test).type(torch.float64)
y_train_tensor = torch.from_numpy(y_train).type(torch.float64).requires_grad_(True) #  Add requires_grad=True
y_test_tensor = torch.from_numpy(y_test).type(torch.float64).requires_grad_(True)  #  Add requires_grad=True


In [ ]:
X_train_tensor.shape

**Build a Simple Neural Model NN**

In [ ]:
class NN():
  def __init__(self , X):
  # Weights and Bais
    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
    self.bias = torch.zeros(1  ,dtype = torch.float64 , requires_grad=True )
  # Forward Pass:
  def forward(self , X):
      z =torch.matmul(X , self.weights) + self.bias
      y_pred = torch.sigmoid(z)
      return y_pred
  # Loss Function
  def loss(self , y_pred , y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred , epsilon, 1-epsilon)
    loss = -torch.mean(y*torch.log(y_pred) + (1-y)*torch.log(1-y_pred))
    return loss


**Hyperparameter**

In [ ]:
lr = 0.1
epochs = 500

In [ ]:
model = NN(X_test_tensor)

In [ ]:
y_pred = model.forward(X_train_tensor)

In [ ]:
train_losses = []
validation_losses = []

for epoch in range(epochs):
    # Calculate Predictions
    y_pred = model.forward(X_train_tensor)

    # Compute Training Loss
    train_loss = model.loss(y_pred, y_train_tensor)

    # Backpropagation
    train_loss.backward()

    # Update Weights and Bias
    with torch.no_grad():
        model.weights -= lr * model.weights.grad
        model.bias -= lr * model.bias.grad

    # Zero Gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    # Store Training Loss
    train_losses.append(train_loss.item())

    # Compute Validation Loss
    with torch.no_grad():
        y_pred_val = model.forward(X_test_tensor)
        val_loss = model.loss(y_pred_val, y_test_tensor).item()
        validation_losses.append(val_loss)

    # Print Loss for Each Epoch
    print(f"Epoch {epoch + 1}, Training Loss: {train_loss.item():.4f}, Validation Loss: {val_loss:.4f}")


**Evaluate**

In [ ]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)  # Get raw predictions
    y_pred = (y_pred > 0.5).float()  # Convert probabilities to binary labels (0 or 1)
    acc = (y_pred == y_test_tensor).float().mean().item()  # Compute accuracy


In [ ]:
acc

In [ ]:
y_pred = model.forward(X_test_tensor[1:10]).argmax(axis= 1)
y_pred

In [ ]:
y_train_tensor[1:10]

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label="Training Loss")
plt.plot(validation_losses, label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.title("Training vs Validation Loss")
plt.show()

In [ ]:
train_loss *100